# AI adoption: denominator walkthrough
An executed walkthrough of the supplied SQLite analysis. Read the case study and metric dictionary alongside this notebook.
AI assisted implementation. Reproduce and understand each result before describing personal ownership.

## Connect to the supplied database

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
assert Path('analysis.sqlite').exists()
db = sqlite3.connect('analysis.sqlite')
print(pd.read_sql_query('SELECT * FROM ai_overview',db).T.to_string(header=False))

respondents            49191.000000
usage_answered         33720.000000
current_users          26469.000000
planned_users           1797.000000
users_answering_trust  26126.000000
low_trust_users         9733.000000
current_adoption           0.784964
low_trust_share            0.372541


## Separate current and planned usage
Blank answers remain visible.

In [2]:
print(pd.read_sql_query("SELECT COALESCE(ai_usage,'Missing answer') AS answer, COUNT(*) AS respondents FROM respondents GROUP BY ai_usage ORDER BY respondents DESC",db).to_string(index=False))

                                     answer  respondents
                  Yes, I use AI tools daily        15883
                             Missing answer        15471
                 Yes, I use AI tools weekly         5958
                    No, and I don't plan to         5454
Yes, I use AI tools monthly or infrequently         4628
                     No, but I plan to soon         1797


## Trust conditional on current use
Neutral answers remain in the denominator.

In [3]:
print(pd.read_sql_query("SELECT COALESCE(trust,'Missing answer') AS trust, COUNT(*) AS current_users FROM respondents WHERE current_user=1 GROUP BY trust ORDER BY current_users DESC",db).to_string(index=False))

                     trust  current_users
            Somewhat trust           9300
         Somewhat distrust           6705
Neither trust nor distrust           6113
           Highly distrust           3028
              Highly trust            980
            Missing answer            343


## Multiple-choice frustrations
One person can contribute to more than one row.

In [4]:
print(pd.read_sql_query('SELECT frustration, respondents, denominator, ROUND(100.0*share,1) AS share_pct FROM frustration_summary',db).to_string(index=False))
db.close()

                                         frustration  respondents  denominator  share_pct
   AI solutions that are almost right, but not quite        18584        24827       74.9
  Debugging AI-generated code is more time-consuming        12043        24827       48.5
I’ve become less confident in my own problem-solving         5744        24827       23.1
   It’s hard to understand how or why the code works         4266        24827       17.2
                                   Other (write in):         2869        24827       11.6
                      I don’t use AI tools regularly         2539        24827       10.2
                  I haven’t encountered any problems         1105        24827        4.5
